In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
path = 'drive/My Drive'

In [9]:
import pandas as pd

data = pd.read_csv(path+'/spotify_synthetic_data.csv')

In [42]:
data.head()

,track_name,artist,album,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,genre
0,Track_1,Artist_52,Album_12,0.808235,0.153784,-24.860740,0.648209,0.575719,0.289405,0.760808,0.155540,130.891275,Hip-Hop
1,Track_2,Artist_93,Album_46,0.947688,0.886171,-59.490265,0.413194,0.583386,0.662430,0.850780,0.963687,128.628400,Jazz
2,Track_3,Artist_15,Album_32,0.072316,0.458519,-14.339641,0.405619,0.749987,0.613122,0.257006,0.946790,110.653032,Hip-Hop
3,Track_4,Artist_72,Album_15,0.955115,0.564591,-17.219142,0.679552,0.509890,0.531003,0.407092,0.336110,199.184331,Hip-Hop
4,Track_5,Artist_61,Album_16,0.522577,0.663027,-38.022817,0.067094,0.104823,0.682964,0.498569,0.639748,75.563707,Electronic


In [43]:
data.tail()

,track_name,artist,album,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,genre
1995,Track_1996,Artist_86,Album_11,0.649379,0.502783,-37.580982,0.865429,0.679873,0.599640,0.054622,0.437652,64.799919,R&B
1996,Track_1997,Artist_51,Album_35,0.668877,0.144436,-36.712713,0.296502,0.642931,0.566667,0.491045,0.855468,186.898119,Pop
1997,Track_1998,Artist_88,Album_8,0.798656,0.442066,-54.258593,0.159283,0.861130,0.654168,0.091417,0.050169,188.583234,Pop
1998,Track_1999,Artist_41,Album_3,0.932753,0.334105,-5.124934,0.435389,0.486426,0.307750,0.159786,0.881962,138.594652,Jazz
1999,Track_2000,Artist_17,Album_14,0.020138,0.521831,-54.537821,0.401899,0.134457,0.126426,0.383038,0.852070,127.491648,Country


In [44]:
import pandas as pd
import hashlib
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [45]:
class SynthetictrackDataset(Dataset):
    def __init__(self, csv_file):

        self.data = pd.read_csv(csv_file)


        self.transformed_data = self._transform_data()

    def _hash_string(self, input_string):

        return hashlib.md5(input_string.encode()).hexdigest()

    def _transform_data(self):

        transformed = pd.DataFrame()


        transformed['user_id'] = self.data['track_name'].apply(lambda x: self._hash_string(x))

        transformed['track_id'] = self.data['album'].apply(lambda x: int(self._hash_string(x), 16) % 100000)


        transformed['rating'] = (self.data['danceability'] + self.data['energy']) / 2 * 5
        transformed['rating'] = transformed['rating'].clip(0.5, 5.0).round(1)
        return transformed

    def __len__(self):
        return len(self.transformed_data)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        row = self.transformed_data.iloc[idx]
        sample = {
            'user_id': row['user_id'],
            'track_id': row['track_id'],
            'rating': row['rating']
        }

        return sample


In [46]:
# Custom Dataset class for encoded data
class trackRatingDataset(Dataset):
    def __init__(self, user_ids, track_ids, ratings):
        self.user_ids = torch.LongTensor(user_ids)
        self.track_ids = torch.LongTensor(track_ids)
        self.ratings = torch.FloatTensor(ratings)

    def __len__(self): #len(train_dataset)
        return len(self.ratings)

    def __getitem__(self, idx): #train_dataset[idx]
        return {
            'user_ids': self.user_ids[idx],
            'track_ids': self.track_ids[idx],
            'ratings': self.ratings[idx]
        }

In [47]:
# Two-Tower Model
class TwoTowerNetwork(nn.Module):
    def __init__(self, num_users, num_tracks, embedding_dim):
        super(TwoTowerNetwork, self).__init__()


        self.user_embedding = nn.Embedding(num_users, embedding_dim)


        self.track_embedding = nn.Embedding(num_tracks, embedding_dim)


        self.output_layer = nn.Linear(1, 1)

    def forward(self, user_ids, track_ids):

        user_embedded = self.user_embedding(user_ids)
        track_embedded = self.track_embedding(track_ids)

        # Compute dot product
        dot_product = torch.sum(user_embedded * track_embedded, dim=1, keepdim=True)


        output = self.output_layer(dot_product)
        return output

In [48]:
# Data preprocessing function
def preprocess_data(csv_file):
    dataset = SynthetictrackDataset(csv_file)
    transformed_data = dataset.transformed_data

    user_encoder = LabelEncoder()
    track_encoder = LabelEncoder()

    transformed_data['user_id_encoded'] = user_encoder.fit_transform(transformed_data['user_id'])
    transformed_data['track_id_encoded'] = track_encoder.fit_transform(transformed_data['track_id'])


    train_df, test_df = train_test_split(transformed_data, test_size=0.2, random_state=42)

    # Create datasets
    train_dataset = trackRatingDataset(
        train_df['user_id_encoded'].values,
        train_df['track_id_encoded'].values,
        train_df['rating'].values
    )

    test_dataset = trackRatingDataset(
        test_df['user_id_encoded'].values,
        test_df['track_id_encoded'].values,
        test_df['rating'].values
    )

    return train_dataset, test_dataset, user_encoder, track_encoder

In [49]:
# Training function
def train_model(model, train_loader, test_loader, num_epochs=5, device='cuda' if torch.cuda.is_available() else 'cpu'):
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters())

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for batch in train_loader:
            user_ids = batch['user_ids'].to(device)
            track_ids = batch['track_ids'].to(device)
            ratings = batch['ratings'].to(device)

            optimizer.zero_grad()
            predictions = model(user_ids, track_ids)
            loss = criterion(predictions.squeeze(), ratings)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in test_loader:
                user_ids = batch['user_ids'].to(device)
                track_ids = batch['track_ids'].to(device)
                ratings = batch['ratings'].to(device)

                predictions = model(user_ids, track_ids)
                val_loss += criterion(predictions.squeeze(), ratings).item()

        print(f'Epoch {epoch+1}/{num_epochs}:')
        print(f'Training Loss: {total_loss/len(train_loader):.4f}')
        print(f'Validation Loss: {val_loss/len(test_loader):.4f}\n')


In [50]:
# Evaluation function
def evaluate_model_for_user(model, user_id, all_track_ids, user_encoder, track_encoder, device='cuda' if torch.cuda.is_available() else 'cpu'):
    """
    Generate predictions for a single user across all tracks.
    """
    model.eval()

    # Encode the user_id to its corresponding integer
    encoded_user_id = user_encoder.transform([user_id])[0]

    # Create a tensor for the user ID repeated for all track IDs
    user_ids_tensor = torch.LongTensor([encoded_user_id] * len(all_track_ids)).to(device)

    # Create a tensor for all track IDs
    track_ids_tensor = torch.LongTensor(all_track_ids).to(device)


    with torch.no_grad():
        predicted_ratings = model(user_ids_tensor, track_ids_tensor).cpu().numpy().flatten()


    results_df = pd.DataFrame({
        'original_user_id': [user_id] * len(all_track_ids),
        'original_track_id': track_encoder.inverse_transform(all_track_ids),
        'predicted_rating': predicted_ratings
    })

    # Sort tracks by predicted rating in descending order
    sorted_results = results_df.sort_values('predicted_rating', ascending=False)

    return sorted_results

In [51]:
dataset = SynthetictrackDataset(path+'/spotify_synthetic_data.csv')
transformed_data = dataset.transformed_data


user_encoder = LabelEncoder()
track_encoder = LabelEncoder()

transformed_data['user_id_encoded'] = user_encoder.fit_transform(transformed_data['user_id'])
transformed_data['track_id_encoded'] = track_encoder.fit_transform(transformed_data['track_id'])


train_df, test_df = train_test_split(transformed_data, test_size=0.2, random_state=42)

# Create datasets
train_dataset = trackRatingDataset(
    train_df['user_id_encoded'].values,
    train_df['track_id_encoded'].values,
    train_df['rating'].values
)

test_dataset = trackRatingDataset(
    test_df['user_id_encoded'].values,
    test_df['track_id_encoded'].values,
    test_df['rating'].values
)

In [52]:
train_dataset[0]

{'user_ids': tensor(1805), 'track_ids': tensor(3), 'ratings': tensor(2.7000)}

In [53]:
transformed_data

,user_id,track_id,rating,user_id_encoded,track_id_encoded
0,f4b5f1d2a8d2e51f6c1c6e97c487446f,44371,2.4,1920,18
1,1e1e4966959e6065401e820c2143f45c,83941,4.6,236,38
2,72a6925f6e28951accc6dd8619154e66,72502,1.3,901,34
3,88bb5dde8a64b703f11de3f2977e1b82,90882,3.8,1081,45
4,07f9a8a3588bc98b7f16e354827cc67e,58788,3.0,71,27
...,...,...,...,...,...
1995,d801efa926388de3ad941b808e25bf20,83703,2.9,1689,37
1996,3bc04ba124f43917b2a7c400d362a2e1,48306,2.0,486,20
1997,51d8348cde481c4e9817ed30cb0144d1,85990,3.1,650,39
1998,90e1ed69be9e8bd523a624b2b21877aa,86110,3.2,1135,40


In [57]:
csv_file = path+'/spotify_synthetic_data.csv'


train_dataset, test_dataset, user_encoder, track_encoder = preprocess_data(csv_file)


train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


num_users = len(user_encoder.classes_)
num_tracks = len(track_encoder.classes_)
embedding_dim = 32
model = TwoTowerNetwork(num_users, num_tracks, embedding_dim)

# Train the model
train_model(model, train_loader, test_loader, num_epochs=10)



# user_id_encoded = test_dataset[0]['user_ids'].item()  # Extract the user ID as an integer
# user_id = user_encoder.inverse_transform([user_id_encoded])[0]

# results = evaluate_model(model, user_id, test_loader, user_encoder, track_encoder)
all_track_ids = test_dataset.track_ids.numpy()

# Evaluate the model for a specific user
user_id_encoded = train_dataset[0]['user_ids'].item()  # Get a valid encoded user ID
user_id = user_encoder.inverse_transform([user_id_encoded])[0]  # Decode to the original user ID

# Predict for this user across all tracks
tracks_sorted = evaluate_model_for_user(model, user_id, all_track_ids, user_encoder, track_encoder).drop_duplicates()

Epoch 1/10:
Training Loss: 19.7997
Validation Loss: 17.4080

Epoch 2/10:
Training Loss: 18.0834
Validation Loss: 16.4174

Epoch 3/10:
Training Loss: 16.5734
Validation Loss: 15.5400

Epoch 4/10:
Training Loss: 15.2570
Validation Loss: 14.7293

Epoch 5/10:
Training Loss: 14.0902
Validation Loss: 13.9829

Epoch 6/10:
Training Loss: 13.0713
Validation Loss: 13.2901

Epoch 7/10:
Training Loss: 12.1469
Validation Loss: 12.6768

Epoch 8/10:
Training Loss: 11.3289
Validation Loss: 12.1215

Epoch 9/10:
Training Loss: 10.6074
Validation Loss: 11.6009

Epoch 10/10:
Training Loss: 9.9587
Validation Loss: 11.1128



In [58]:
print(tracks_sorted.head())

                     original_user_id  original_track_id  predicted_rating
174  e6334bdf4264c535c33c89e701c9d53c              45263          5.335276
379  e6334bdf4264c535c33c89e701c9d53c              20552          4.686751
306  e6334bdf4264c535c33c89e701c9d53c              60100          3.873894
60   e6334bdf4264c535c33c89e701c9d53c              33767          3.336124
239  e6334bdf4264c535c33c89e701c9d53c               7221          3.032466


In [56]:
print(tracks_sorted.tail())

                     original_user_id  original_track_id  predicted_rating
66   e6334bdf4264c535c33c89e701c9d53c              23796         -2.911472
87   e6334bdf4264c535c33c89e701c9d53c               7221         -3.397246
199  e6334bdf4264c535c33c89e701c9d53c              42186         -3.905228
222  e6334bdf4264c535c33c89e701c9d53c              86110         -4.173895
17   e6334bdf4264c535c33c89e701c9d53c              99678         -5.370040


In [31]:
tracks_sorted

,original_user_id,original_track_id,predicted_rating
200,e6334bdf4264c535c33c89e701c9d53c,37318,4.312819
113,e6334bdf4264c535c33c89e701c9d53c,13691,4.016983
109,e6334bdf4264c535c33c89e701c9d53c,88371,3.709528
39,e6334bdf4264c535c33c89e701c9d53c,93133,3.461681
9,e6334bdf4264c535c33c89e701c9d53c,72502,3.365758
132,e6334bdf4264c535c33c89e701c9d53c,60100,2.989783
63,e6334bdf4264c535c33c89e701c9d53c,63964,2.826257
70,e6334bdf4264c535c33c89e701c9d53c,57213,2.660081
87,e6334bdf4264c535c33c89e701c9d53c,7221,2.653437
79,e6334bdf4264c535c33c89e701c9d53c,50173,2.206160
